In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
from scipy import io
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)])
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

In [ ]:
labelDict = io.loadmat('/home/austin/Aggression/Analysis/OriginalModel/Windows.mat')
mouse = np.squeeze(labelDict['mouse'])
group = np.squeeze(labelDict['group'])
time = np.squeeze(labelDict['time'])
condition = np.squeeze(labelDict['condition'])
behavior = np.squeeze(labelDict['behavior'])

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

In [ ]:
power,coherence,granger,labels = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',
                                          fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6 

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X = np.hstack((power,coherence,granger))


In [ ]:
labels['windows'].keys()

In [ ]:
labelDict = labels['windows']
mouse = np.squeeze(labelDict['mouse'])
group = np.squeeze(labelDict['group'])
time = np.squeeze(labelDict['time'])
condition = np.squeeze(labelDict['condition'])
behavior = np.squeeze(labelDict['behavior'])

In [ ]:
len(mouse)

In [ ]:
mu = 0.1
dirName = '/home/austin/Aggression/Experiments/NewFeatures/Limited/Unbalanced'
sname = dirName + '/checkpoints/Model_enc_' + str(int(100*mu))

In [ ]:
n_components=8
encoder = keras.Sequential([
                keras.layers.InputLayer(input_shape=X.shape[1]),
                keras.layers.Dense(60,activation='elu'),
                keras.layers.Dense(n_components,activation='softplus')
        ])
encoder.load_weights(sname)

In [ ]:
S_ = encoder(X)
S_test = S_.numpy()

In [ ]:
model_name = dirName + '/Unbalanced_Joint_12_enc_' + str(int(100*mu)) + '.p'
print(model_name)
model_dict = pickle.load(open(model_name,'rb'))

In [ ]:
components = model_dict['components']
X_recon = np.dot(S_test,components)
np.mean((X-X_recon)**2)

In [ ]:
np.mean(X**2)

In [ ]:
np.mean((X-np.mean(X,axis=0))**2)

In [ ]:
print('>>>>>>>>>>>>>')
print(model_dict['recon_random_train'])
print(model_dict['recon_random_test'])
print('>>>>>>>>>>>>>')

print(model_dict['recon_sae_train'])
print(model_dict['recon_sae_test'])
print('>>>>>>>>>>>>>')

In [ ]:
Scores_supervised_neg = S_test[:,0]
Scores_supervised_pos = S_test[:,1]
Scores_unsupervised = S_test[:,2:]

In [ ]:
y = np.zeros(len(mouse))
idx_pos = (condition==4)&(behavior==1)
y[idx_pos] = 1

In [ ]:
idx_neg = ((behavior==2)&((condition==4)|(condition==6)|(condition==8)))
idx_tot = idx_pos|idx_neg

In [ ]:
Scores_supervised_neg_limited = Scores_supervised_neg[idx_tot]
Scores_supervised_pos_limited = Scores_supervised_pos[idx_tot]
Scores_unsupervised_limited = Scores_unsupervised[idx_tot]
y_limited = y[idx_tot]
mouse_limited = mouse[idx_tot]
print(np.sum(idx_tot))

In [ ]:
mice = np.unique(mouse_limited)
nMice = len(mice)

In [ ]:
auc_sup_neg = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_neg_limited[idx_mouse]
    auc_sup_neg[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sup_neg[i]))
mean = np.mean(auc_sup_neg)
ci = np.std(auc_sup_neg)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

In [ ]:
auc_sup_pos = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_pos_limited[idx_mouse]
    auc_sup_pos[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sup_pos[i]))
mean = np.mean(auc_sup_pos)
ci = np.std(auc_sup_pos)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

## Behavior = 1

In [ ]:
idx_neg = ((behavior==0)&((condition==4)|(condition==6)|(condition==8)))
idx_tot = idx_pos|idx_neg

Scores_supervised_neg_limited = Scores_supervised_neg[idx_tot]
Scores_supervised_pos_limited = Scores_supervised_pos[idx_tot]
Scores_unsupervised_limited = Scores_unsupervised[idx_tot]
y_limited = y[idx_tot]
mouse_limited = mouse[idx_tot]

In [ ]:
mice = np.unique(mouse_limited)
nMice = len(mice)

auc_sup_pos = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_pos_limited[idx_mouse]
    auc_sup_pos[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sup_pos[i]))
mean = np.mean(auc_sup_pos)
ci = np.std(auc_sup_pos)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

In [ ]:
auc_sup_neg = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_neg_limited[idx_mouse]
    auc_sup_neg[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sup_neg[i]))
mean = np.mean(auc_sup_neg)
ci = np.std(auc_sup_neg)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

In [ ]:
auc_unsup = np.zeros((nMice,6))
for i in range(nMice):
    for j in range(6):
        idx_mouse = mouse_limited==mice[i]
        y_true = y_limited[idx_mouse]
        y_hat = Scores_unsupervised_limited[idx_mouse,j]
        auc_unsup[i,j] = roc_auc_score(y_true,y_hat)
means = np.mean(auc_unsup,axis=0)
stds = np.std(auc_unsup,axis=0)
ci = 1.96*stds/np.sqrt(nMice)

for j in range(6):
    mainStr = 'Test set AUC CI Factor %d: %0.2f +- %0.2f'%(j+1,means[j],ci[j])
    print(mainStr)